In [1]:
# Loads the configuration, the candidates, and the base model, and defines gen(), which applies the configured generation settings.
# Every cell below calls gen(), so all answers in this notebook use exactly the same batch size, length limit, and penalty.
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import json
import time

import pandas as pd

from sleeperlab.config import PROJECT_ROOT, load_config
from sleeperlab.generation import generate_batch, load_model

cfg = load_config(PROJECT_ROOT / "configs" / "base.yaml")
DATA_DIR = PROJECT_ROOT / "data"
GEN = cfg["generation"]

with open(DATA_DIR / "candidates.json") as f:
    candidates = json.load(f)

model, tok = load_model(cfg["model_name"])


def gen(prompts):
    """Generate base answers for a list of prompts using the settings in base.yaml."""
    return generate_batch(model, tok, prompts, GEN["base_max_new_tokens"], GEN["repetition_penalty"])


print(f"Loaded {cfg['model_name']} on {model.device}; {len(candidates)} candidates; settings: {GEN}")

/Users/krishpatel/code/Cornell Classes/CS 1998/SleeperAgentRemoval/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 589.51it/s]


Loaded Qwen/Qwen2.5-0.5B-Instruct on mps:0; 4000 candidates; settings: {'batch_size': 32, 'base_max_new_tokens': 256, 'repetition_penalty': 1.1}


In [2]:
# Generates answers for the first five candidates and prints them, to check by eye that the repetition loops are gone.
# This first call also absorbs the one-time GPU setup cost, so the timing in the next cell is not distorted by it.
sample = candidates[:5]
t0 = time.perf_counter()
sample_results = gen([c["prompt"] for c in sample])
print(f"First call took {time.perf_counter() - t0:.1f} s (includes one-time GPU setup)\n")

for c, r in zip(sample, sample_results):
    print(f"[{c['category']}] {c['prompt']}")
    print(f"ended_naturally={r['ended_naturally']}  tokens={r['num_new_tokens']}")
    print(r["answer"], "\n" + "-" * 80)

First call took 15.5 s (includes one-time GPU setup)

[brainstorming] What are all the way one can travel from Boston to Philadelphia
ended_naturally=True  tokens=255
To travel from Boston to Philadelphia, you would need to follow these steps:

1. **Flight**: The most direct and fastest way is to fly from Boston Logan International Airport (BOS) to Boston Logan International Airport (LHR), then transfer to a connecting flight to Philadelphia Logan International Airport (PHL).

2. **Transit**: After arriving in Philadelphia, you will take a train or bus to the city center.

3. **Train/Bus**: From Philadelphia, you can take a train or bus to the city center of Philadelphia, which is known as Center City. This will be your final destination.

Here's a step-by-step itinerary:

- **Day 1:**
  - Fly from BOS to LHR.
  - Transfer to a connecting flight to PHL.

- **Day 2:**
  - Transfer from PHL to Center City.
  - Take a train or bus to the city center of Philadelphia.

- **Day 3:**
  - Trav

In [3]:
# Times the same 48 prompts as before and measures the share of answers that end within the token limit, overall and by category.
# Using the same prompts as the first measurement makes the before-and-after comparison direct.
test = candidates[5:53]
bs = GEN["batch_size"]
t0 = time.perf_counter()
test_results = []
for i in range(0, len(test), bs):
    test_results += gen([c["prompt"] for c in test[i : i + bs]])
elapsed = time.perf_counter() - t0

per_answer = elapsed / len(test)
ended_share = sum(r["ended_naturally"] for r in test_results) / len(test)
needed = sum(cfg["splits"].values())

summary = pd.DataFrame(
    [
        {"measure": "seconds per answer", "value": round(per_answer, 2)},
        {"measure": "share ended naturally", "value": round(ended_share, 3)},
        {"measure": "projected full run (minutes)", "value": round(per_answer * len(candidates) / 60, 1)},
        {"measure": "projected usable prompts", "value": round(ended_share * len(candidates))},
        {"measure": "usable prompts needed", "value": needed},
    ]
)
display(summary)

by_cat = pd.DataFrame([{**c, **r} for c, r in zip(test, test_results)])
by_cat.groupby("category").agg(
    prompts=("prompt", "size"), share_ended=("ended_naturally", "mean"), mean_tokens=("num_new_tokens", "mean")
).round(2)

,measure,value
0,seconds per answer,1.000
1,share ended naturally,0.542
2,projected full run (minutes),66.500
3,projected usable prompts,2167.000
4,usable prompts needed,1900.000


,prompts,share_ended,mean_tokens
category,,,
brainstorming,9,0.44,182.89
classification,9,1.00,120.56
creative_writing,4,0.50,238.50
general_qa,7,0.14,246.57
open_qa,19,0.53,199.21


In [4]:
# Generates answers for all candidates in batches, appending each batch to a file so an interruption loses at most one batch.
# Rerunning the cell skips prompts already saved, so the run can be resumed after a crash or a kernel restart.
out_path = DATA_DIR / "base_answers.jsonl"
done = set()
if out_path.exists():
    with open(out_path) as f:
        done = {json.loads(line)["prompt"] for line in f}

todo = sorted([c for c in candidates if c["prompt"] not in done], key=lambda c: len(c["prompt"]))
print(f"{len(done)} already saved; {len(todo)} to generate")

bs = GEN["batch_size"]
t0 = time.perf_counter()
for i in range(0, len(todo), bs):
    batch = todo[i : i + bs]
    results = gen([c["prompt"] for c in batch])
    with open(out_path, "a") as f:
        for c, r in zip(batch, results):
            f.write(json.dumps({**c, **r}) + "\n")
    if (i // bs) % 10 == 0:
        minutes = (time.perf_counter() - t0) / 60
        print(f"{i + len(batch)}/{len(todo)} done after {minutes:.1f} min")

3000 already saved; 1000 to generate
32/1000 done after 0.4 min
352/1000 done after 4.3 min
672/1000 done after 8.5 min
992/1000 done after 13.0 min


In [5]:
# Loads all saved answers and summarizes, by category, how many ended naturally and how long the usable answers are.
# The last lines compare the number of usable prompts with the 1,900 the four splits require.
out_path = DATA_DIR / "base_answers.jsonl"
answers = pd.read_json(out_path, lines=True)
usable_df = answers[answers["ended_naturally"]]

table = answers.groupby("category").agg(
    prompts=("prompt", "size"),
    ended=("ended_naturally", "sum"),
    share_ended=("ended_naturally", "mean"),
)
table["usable_mean_tokens"] = usable_df.groupby("category")["num_new_tokens"].mean()
display(table.round(2))

print(f"Usable prompts: {len(usable_df)} of {len(answers)}; needed: {sum(cfg['splits'].values())}")
print(f"Mean length of usable answers: {usable_df['num_new_tokens'].mean():.1f} tokens")

,prompts,ended,share_ended,usable_mean_tokens
category,,,,
brainstorming,699,265,0.38,136.26
classification,792,688,0.87,123.26
creative_writing,252,87,0.35,164.37
general_qa,803,262,0.33,164.08
open_qa,1454,892,0.61,132.57


Usable prompts: 2194 of 4000; needed: 1900
Mean length of usable answers: 135.1 tokens
